# Create Beijing Municipal Natural Science Foundation Awards

Creates awards from the 北京市自然科学基金委员会办公室 (基金办) per-project rosters
published on kw.beijing.gov.cn (live site + Internet Archive raw captures).

**Prerequisites:** run `scripts/local/beijing_nsf_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/beijing_nsf/beijing_nsf_projects.parquet`.

**Data source / routing (how-to §2.3.2):** kw.beijing.gov.cn is shared with the
Beijing Municipal S&T Commission. Only 基金办 rosters are used here: the 2022 资助项目
xlsx (办事结果); 2022/2023 资助决定 and 拟资助 lists for 杰出青年 / 重点研究专题 / 面上 /
青年 / 小米·海淀·丰台·昌平 联合基金 / 京津冀基础研究合作专项; and the 2020 第二批验收清单
(completed 2017-2018 projects). The BMSTC 立项公开清单 go to `beijing_bmstc`
(priority 510). Since 2024 the rosters are behind the nsf.kw.beijing.gov.cn SSO, so
coverage is the public 2017-2023 lists only. Filter: none needed -- every roster row
is a research grant (no subsidy lines on these lists).

**funder_award_id (§2.1.1):** the published 资助编号 (7202106 / JQ22001 / Z230001 /
L233001 / J230001), which is the form citing works carry
(`works?filter=grants.funder:F4320334977&select=grants` -> 7164309, L243009, 4232058 ...).
Rows whose roster prints no number (most of the 2022 xlsx) use the stable synthetic
key `BJNSF-<sha1(title|institution)[:12]>`.

**Amounts:** 万元 converted to CNY in the script; NULL where the roster has none
(the 2022 xlsx), so §6.7 amount coverage is partial by design.

**PI names:** Chinese, family-first -> `family_name`, `given_name` NULL.

**Funder details (Path A, F4320*):**
- funder_id: `4320334977` (Beijing Municipal Natural Science Foundation, CN)
- TWIN: `4320322919` "Natural Science Foundation of Beijing Municipality" is the same
  body -- flagged to the coordinator (oxjob #1451), not used here.

**Priority:** `523` (direct funder ingest; higher wins under DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.beijing_nsf_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/beijing_nsf/beijing_nsf_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.beijing_nsf_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.beijing_nsf_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.beijing_nsf_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320334977 is a Crossref-registered F4320* funder, so it MUST resolve to
exactly 1 row in `openalex.funders.funders`. If 0 rows, STOP and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320334977;

## Step 2: Create Beijing Municipal Natural Science Foundation Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.beijing_nsf_awards
USING delta
AS
WITH
-- Path A: F4320334977 is F4320* (Crossref-registered) -> canonical metadata from the dim.
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320334977
),

awards_transformed AS (
    SELECT
        -- funder_award_id is always populated by the script (published number, or the
        -- stable synthetic key documented in the header), so the id hashes on it alone.
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,

        g.display_name as display_name,
        CAST(NULL AS STRING) as description,

        f.funder_id,
        NULLIF(TRIM(g.funder_award_id), '') as funder_award_id,

        -- the script already converted 万元 -> CNY (x10,000)
        TRY_CAST(g.amount AS DOUBLE) as amount,
        CASE WHEN TRY_CAST(g.amount AS DOUBLE) IS NOT NULL THEN 'CNY' ELSE NULL END as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        -- Funding type from the scheme label (杰青/青年 -> fellowship; 重点/联合/合作 -> research; else grant).
        CASE
            WHEN g.funder_scheme LIKE '%杰出青年%' THEN 'fellowship'
            WHEN g.funder_scheme LIKE '%青年%' THEN 'fellowship'
            WHEN g.funder_scheme LIKE '%重点%' THEN 'research'
            WHEN g.funder_scheme LIKE '%联合%' THEN 'research'
            WHEN g.funder_scheme LIKE '%合作%' THEN 'research'
            ELSE 'grant'
        END as funding_type,

        NULLIF(TRIM(g.funder_scheme), '') as funder_scheme,

        'beijing_nsf' as provenance,

        TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
        TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
        COALESCE(YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')), TRY_CAST(g.start_year AS INT)) as start_year,
        YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) as end_year,

        -- Lead investigator: Chinese PI full name in family_name, given NULL (NSFC precedent).
        CASE
            WHEN (g.lead_family_name IS NOT NULL AND TRIM(g.lead_family_name) != '')
              OR (g.institution IS NOT NULL AND TRIM(g.institution) != '') THEN
                struct(
                    CAST(NULL AS STRING) as given_name,
                    NULLIF(TRIM(g.lead_family_name), '') as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        NULLIF(TRIM(g.institution), '') as name,
                        'China' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        g.landing_page_url as landing_page_url,

        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM openalex.awards.beijing_nsf_raw g
    CROSS JOIN src_funder f
    WHERE g.display_name IS NOT NULL
      AND TRIM(g.display_name) != ''
      AND g.funder_award_id IS NOT NULL
      AND TRIM(g.funder_award_id) != ''
)

SELECT * FROM awards_transformed;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'beijing_nsf' AND priority = 523;

-- Insert into openalex_awards_raw with priority.
-- Priority 523: direct-from-funder ingest. Higher wins under the
-- 2026-06-20 DESC dedup (oxjob #500), so it outranks acknowledgement shells
-- (priority 0) and grant-DOI stubs (priority 1).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    523 as priority
FROM openalex.awards.beijing_nsf_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_beijing_nsf_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.beijing_nsf_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, amount, currency,
       start_date, end_date, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.beijing_nsf_awards LIMIT 20;

In [ ]:
%sql
SELECT funding_type, funder_scheme, COUNT(*) as cnt FROM openalex.awards.beijing_nsf_awards
GROUP BY funding_type, funder_scheme ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt FROM openalex.awards.beijing_nsf_awards
GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- §6.3 / §6.7 coverage
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(currency) as has_currency,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.beijing_nsf_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (§6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'beijing_nsf'
GROUP BY provenance, priority;